In [1]:
import os

import pandas as pd

# Fix NIBIO AGRITIL data v20260106

In January 2026, NIBIO delivered a entirely new set of AGRITIL output for the period from 2013 to 2024. There are significant differences in this dataset compared to the previous version, and Sigrun later discovered an error with the REGINE ordering in the AGRITIL-SPC output. See e-mail discussion on 27.08.2026 for details.

NIBIO will fix the problem at their end, but as a temporary solution this notebook attempts to post-fix the original NIBIO data delivery.

In [2]:
st_yr, end_yr = 2013, 2024
version = "20260106"
base_dir = r"/home/jovyan/shared/common/teotil3/agri_data"

In [3]:
# REGINE order for AGRTIL-SPC output. Provided by Sigrun 27.08.2026
spc_order_xlsx = os.path.join(
    base_dir, f"bad_raw_agri_data_v{version}", "Regine_SPC_sortering.xlsx"
)
spc_order = pd.read_excel(spc_order_xlsx, sheet_name="Ark1", index_col=0)
spc_order.index.name = None

# Loop over original files from Franziska
for year in range(st_yr, end_yr + 1):
    # Tidied output file to create
    out_xlsx = os.path.join(
        base_dir, f"agri_data_v{version}", f"nibio_agri_data_{year}.xlsx"
    )
    with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
        for sheet in ["Annual loss", "Risk loss"]:
            # Read original data from Franziska with incorrect SPC ordering
            raw_xlsx = os.path.join(
                base_dir,
                f"bad_raw_agri_data_v{version}",
                f"Leveranse_{year}_{version}.xlsx",
            )
            df = pd.read_excel(raw_xlsx, sheet_name=sheet, header=(0, 1), index_col=0)

            # Separate N and SPC data
            n_cols = [col for col in df.columns if col[0].startswith(("TN", "NO3-N"))]
            spc_cols = [col for col in df.columns if col not in n_cols]
            n_df = df[n_cols].copy()
            spc_df = df[spc_cols].copy()

            # Reindex SPC data
            assert len(spc_order) == len(spc_df)
            spc_df.index = spc_order.index

            # Merge
            df = pd.concat([n_df, spc_df], axis="columns")

            # Save
            df.to_excel(writer, sheet_name=sheet)
            writer.sheets[sheet].delete_rows(3)